In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_docs = Document(
    page_content="Hello World @",
    metadata = {"source" : "http//www.google.com"}
)

In [4]:
sample_docs

Document(metadata={'source': 'http//www.google.com'}, page_content='Hello World @')

In [5]:
# Text data
from langchain_community.document_loaders.text import TextLoader

C:\Users\Hemant\AppData\Local\Temp\ipykernel_22028\3188525453.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [54]:
loader = TextLoader(
    "Data/Python.txt",
    encoding = "utf-8"
)

In [7]:
document = loader.load()

In [8]:
document

[Document(metadata={'source': 'Data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [9]:
# PDF Data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# from langchain_community.document_loaders.pdf import PyMuPDFLoader

In [10]:
# pdf_loader = PyPDFLoader(
#     "Data/research.pdf"
# )

In [11]:
# document = pdf_loader.load()

In [12]:
# document

## Ingestion Pipeline

In [13]:
# Data => Document

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Document

In [14]:
# pdf => docs conversion function
def load_all_pdfs():
    folder_path = "Data/pdfs"
    docs = []
    num_of_docs = 0

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete pdf name
            pdf_path = os.path.join(folder_path , filename)
            
            pdf_loader = PyPDFLoader(pdf_path)

            doc = pdf_loader.load()

            docs.extend(doc)

            num_of_docs += 1

    print(f" number of docs : {num_of_docs}")
    print(f"total pages : {len(docs)}")
    return docs

In [15]:
all_pdf_docs = load_all_pdfs()

 number of docs : 2
total pages : 32


In [16]:
# Chunks

!pip install langchain_text_splitters

### Chunks

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(document , chunk_size = 500 , chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_documents = text_splitter.split_documents(document)
    return chunked_documents

In [18]:
chunks = split_docs(all_pdf_docs)

In [19]:
type(all_pdf_docs)

list

In [20]:
len(chunks)

321

### Embedding

In [21]:
from sentence_transformers import SentenceTransformer

In [22]:
class EmbeddingManager :
    def __init__(self , model_name = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        print(f"loading model ...  {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"embedding dimensions : {self.model.get_embedding_dimension()}")

    def generate_embedding(self , text):
        embeddings = self.model.encode(text , show_progress_bar = True)
        print("embedding shape : ",embeddings.shape)
        return embeddings

In [23]:
embedding = EmbeddingManager()

loading model ...  all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions : 384


### Vector DB

In [24]:
import chromadb
import uuid

In [25]:
class VectorStoreManager:
    def __init__(self , persist_directory = "data/vector_store" , collection_name = "pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name 
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory , exist_ok = True)

        self.client = chromadb.PersistentClient(path = self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description" : "vector store collection for pdf embeddings in docs"}
        )

        print("initialized the vector store with collection : " , self.collection_name)
        print("docs in collections : " , self.collection.count())

    def add_document(self , documents , embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("len of docs is not equal to len of embeddings")

        # id , embedding , docs , metadata
        ids = []
        docs_content = []
        all_metadata = []
        embeddings_list = []

        for i , (doc , embedding) in enumerate(zip(documents , embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            all_metadata.append(metadata)

            docs_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids = ids,
                metadatas = all_metadata,
                documents = docs_content,
                embeddings = embeddings_list
            )

        print(f"total documents added in collections : {len(docs_content)}")
        print("docs in collections : " , self.collection.count())

In [26]:
vector_store = VectorStoreManager()

initialized the vector store with collection :  pdf_documents
docs in collections :  321


In [27]:
text = [doc.page_content for doc in chunks]

embeddings = embedding.generate_embedding(text)

vector_store.add_document(chunks , embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embedding shape :  (321, 384)
total documents added in collections : 321
docs in collections :  642


### Retrieve

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [41]:
class RAGRetriever:
    def __init__(self , embeddings_manager , vector_store):
        self.embeddings_manager = embeddings_manager
        self.vector_store = vector_store

    def retrieve(self , query , top_k = 5 , threshold = 0.0):
        query_embeddings = self.embeddings_manager.generate_embedding([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i , (doc_id , meta , document , distance) in enumerate(zip(ids , metadatas , documents , distances)):
                similarity_score = 1 - distance

                if similarity_score >= threshold:
                    retrieved_docs.append({
                        "id" : doc_id,
                        "metadatas" : meta,
                        "documents" : document,
                        "distances" : distance,
                        "similarity_score" : similarity_score,
                        "rank" : i+1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else :
            print("no document")

        return retrieved_docs

In [42]:
rag_retriever = RAGRetriever(embedding , vector_store)

In [43]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape :  (1, 384)
retrieved 5 documents


[{'id': 'doc_c4275e65-14b4-4d71-8e88-3e43d98c2176',
  'metadatas': {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'title': '',
   'source': 'Data/pdfs\\research2.pdf',
   'producer': 'pdfTeX-1.40.25',
   'doc_index': 88,
   'page_label': '1',
   'creator': 'LaTeX with hyperref',
   'content_length': 288,
   'subject': '',
   'keywords': '',
   'page': 0,
   'total_pages': 21,
   'moddate': '2024-03-28T00:54:45+00:00',
   'trapped': '/False',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'author': ''},
  'documents': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'distances': 0.4629150927066803,
  'similarity_score': 0.5370849072933197,
  'rank': 1},
 {'id': 'doc_67c6f5

### Groq API

In [63]:
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [46]:
!pip install langchain-groq

  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)

   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   -------------------- ------------------- 1/2 [langchain-groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [51]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = GROQ_API_KEY,
    model = "qwen/qwen3-32b",
    temperature = 0.1,
    max_tokens = 1024
)

In [57]:
# generate our retrieval-augmemted output
def generate_output(query , retriever , llm , top_k = 3):
    results = retriever.retrieve(query , top_k)

    context = "\n".join([doc["documents"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query

    prompt = f""" use given context to generate the answer for the query
                Context : {context}
                Query : {query}"""

    response = llm.invoke([prompt.format(context = context , query = query)])
    return response.content

In [61]:
answer = generate_output("What is RAG ?" , rag_retriever , llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape :  (1, 384)
retrieved 3 documents


In [62]:
print(answer)

<think>
Okay, the user is asking, "What is RAG?" Let me look at the provided context to form an answer.

First, the context mentions that the survey presents a thorough review of state-of-the-art RAG methods. It talks about the evolution through paradigms like naive RAG and an effective RAG framework. Also, there's a section on evaluation methods covering tasks, datasets, metrics, and benchmarks. The paper's structure includes an introduction to RAG's main concept and current paradigms in Section II.

So, RAG stands for Retrieval-Augmented Generation. From the context, it's a method that combines retrieval of information with generation. The naive RAG is an initial approach, and there's an effective framework mentioned. The survey outlines its evolution and evaluation aspects.

I need to define RAG clearly, mention its purpose, and perhaps touch on its paradigms as per the context. Also, note that it's used in NLP tasks to enhance model responses by retrieving relevant information befo